In [1]:
import pandas as pd

DRIVE_URL = "/content/drive/MyDrive/DORE_tesi_magistrale/dataset"

In [2]:
#@title Functions
def get_nodes(df: pd.DataFrame):
  df = preprocess_dataset(df)
  nodes = pd.concat([df['head'], df['tail']], axis=0)
  nodes = nodes.dropna().drop_duplicates().reset_index(drop=True)
  return pd.Series(nodes)

def preprocess_dataset(df: pd.DataFrame):
	df = df.applymap(lambda x: str(x))
	df = df.applymap(lambda x: x.lower())
	df = df.applymap(lambda x: x.strip())
	df = df.applymap(lambda x: x.strip("."))
	df = df.dropna().drop_duplicates().sample(frac=1).reset_index(drop=True)
	return df

def clean_roberta_df(df: pd.DataFrame):
  df = df[['head', 'relation', 'tail']]
  df['relation'] = df['relation'].replace({'__label__Support': 'support', '__label__Attack': 'attack'})
  df = df[df['relation'] != '__label__noRel']
  return preprocess_dataset(df)

**Loading datasets**

In [3]:
original = pd.read_csv(f"{DRIVE_URL}/original_dataset.csv")
train_df = pd.read_csv(f"{DRIVE_URL}/roberta_dataset/train_relations.tsv", header=None, index_col=False, sep="\t", names=['relation', 'head', 'tail'])
dev_df = pd.read_csv(f"{DRIVE_URL}/roberta_dataset/dev_relations.tsv", header=None, index_col=False, sep="\t", names=['relation', 'head', 'tail'])
test_df = pd.read_csv(f"{DRIVE_URL}/roberta_dataset/test_relations.tsv", header=None, index_col=False, sep="\t", names=['relation', 'head', 'tail'])

In [4]:
print(original['RelationType'].value_counts())
original.head(5)

Support       21689
Attack         3835
Equivalent      706
Name: RelationType, dtype: int64


,Year,date,Dependent,D_type,Speaker1,Governor,G_type,Speaker2,RelationType,long_date
0,1960,07 10,"As a matter of fact in his book, The Strategy ...",Claim,NIXON,For me to have made such a statement would bee...,Claim,NIXON,Support,07-10-1960
1,1960,07 10,"Now I'm very surprised that Senator Kennedy, w...",Claim,NIXON,"As a matter of fact in his book, The Strategy ...",Claim,NIXON,Support,07-10-1960
2,1960,07 10,"Now I'm very surprised that Senator Kennedy, w...",Premise,NIXON,Senator Kennedy also indicated with regard to ...,Claim,NIXON,Support,07-10-1960
3,1960,07 10,"I look at Cuba today, I believe that we are fo...",Claim,NIXON,We think that's pretty good progress,Claim,NIXON,Support,07-10-1960
4,1960,07 10,"I look at Cuba today, I believe that we are fo...",Premise,NIXON,a course which is difficult,Claim,NIXON,Attack,07-10-1960


In [5]:
train_df.head(5)

,relation,head,tail
0,__label__Support,"if a state gets in trouble, well, we can step ...",Let states do this
1,__label__Support,"General Shinsheki, the Army chief of staff, sa...",They avoided even the advice of their own general
2,__label__Support,We think it is appropriate to return to the Am...,The fact is that the program that we put toget...
3,__label__noRel,We will not make war inevitable,"what's the message going to be: ""Please join u..."
4,__label__Support,I believe we have to get ISIS.,We have to worry about ISIS before we can get ...


**Get equivalent relations that were dropped from roberta dataset**

In [6]:
equivalent = original[original['RelationType']=='Equivalent'].rename({'Governor':'head', 'Dependent':'tail', 'RelationType': 'relation'}, axis=1)[['head', 'relation', 'tail']]
equivalent = preprocess_dataset(equivalent)
equivalent_nodes = get_nodes(equivalent)

equivalent.head()

,head,relation,tail
0,"when you tried to act holier than thou, it rea...",equivalent,it really doesn't
1,governor clinton's philosophy is isolate them,equivalent,to do what the congress and governor clinton i...
2,we should know it,equivalent,the american people should be told the facts
3,"nobody has business doing what i just said, do...",equivalent,nobody has that
4,that’s never happened before in america,equivalent,that’s never happened before in america


**Preparing training**

In [7]:
dataset = pd.concat([pd.concat([train_df, dev_df], axis=0), test_df], axis=0).dropna().drop_duplicates().reset_index(drop=True)
print(len(dataset))
dataset.head()

45816


,relation,head,tail
0,__label__Support,"if a state gets in trouble, well, we can step ...",Let states do this
1,__label__Support,"General Shinsheki, the Army chief of staff, sa...",They avoided even the advice of their own general
2,__label__Support,We think it is appropriate to return to the Am...,The fact is that the program that we put toget...
3,__label__noRel,We will not make war inevitable,"what's the message going to be: ""Please join u..."
4,__label__Support,I believe we have to get ISIS.,We have to worry about ISIS before we can get ...


In [8]:
nodes = get_nodes(dataset)
print(len(nodes))
nodes = pd.concat([nodes, equivalent_nodes], axis=0)
print(len(nodes))

31068
32058


In [9]:
nodes.head()

0                       your interest rates will go up
1    stricter interpretation and consolidation of t...
2                                  i did a study on it
3    in fact, is promoting some of these spending p...
4    not to discourage people to go out and discove...
dtype: object

**Create node's type**

In [10]:
# nodes type
nodes_type = pd.concat([original[['Dependent', 'D_type']].rename({'Dependent':'node', 'D_type':'type'}, axis=1), \
                        original[['Governor', 'G_type']].rename({'Governor':'node', 'G_type':'type'}, axis=1)], axis=0)
nodes_type = preprocess_dataset(nodes_type)

In [11]:
nodes_type = pd.DataFrame(nodes_type[nodes_type['node'].isin(nodes)])

In [12]:
nodes_type['relation'] = 'is a'
nodes_type = nodes_type.rename({'node': 'head', 'type':'tail'}, axis=1)[['head', 'relation', 'tail']]
nodes_type.head()

,head,relation,tail
0,we've developed oil and natural gas,is a,premise
1,you have to cut the value of each check by 1 o...,is a,claim
2,the first lady has done great work with an org...,is a,claim
3,don't forget what he tried to do with health care,is a,premise
4,this is public ethics,is a,premise


**Create node's year**

In [13]:
nodes_year = pd.concat([original[['Dependent', 'long_date']].rename({'Dependent':'node', 'long_date':'date'}, axis=1), \
                        original[['Governor', 'long_date']].rename({'Governor':'node', 'long_date':'date'}, axis=1)], axis=0)
nodes_year = preprocess_dataset(nodes_year)

In [14]:
nodes_year = pd.DataFrame(nodes_year[nodes_year['node'].isin(nodes)])

In [15]:
nodes_year['relation'] = 'said in'
nodes_year = nodes_year.rename({'node': 'head', 'date':'tail'}, axis=1)[['head', 'relation', 'tail']]
nodes_year.head()

,head,relation,tail
0,what are our values?,said in,25-09-1988
1,this is an arms escalation,said in,21-10-1984
2,i’ve gotten to know the people of the country ...,said in,09-10-2016
3,i propose $2 billion worth,said in,11-10-2000
4,and 95 percent of the people in the united sta...,said in,02-10-2008


**Cleaning original dataset**

In [16]:
train_df = clean_roberta_df(train_df)
test_df = clean_roberta_df(test_df)
dev_df = clean_roberta_df(dev_df)

In [17]:
train_df.head()

,head,relation,tail
0,he's not been a maverick when it comes to educ...,support,he has not supported tax cuts and significant ...
1,and the wic program,support,i've reached out to people all my life
2,not ideologically driven efforts to push peopl...,attack,"if americans trust me with the presidency, i c..."
3,i don't like that term,attack,"i want to give seniors who are, well, the near..."
4,you must be tough and smart,support,it takes more than that


In [18]:
train_df['relation'].value_counts()

support    16054
attack      2408
Name: relation, dtype: int64

**Augmenting training dataset**

In [19]:
train_df = pd.concat([train_df, equivalent], axis=0)
train_df = pd.concat([train_df, nodes_year], axis=0)
train_df = pd.concat([train_df, nodes_type], axis=0)
train_df = preprocess_dataset(train_df).dropna().drop_duplicates().reset_index(drop=True)

print(train_df['relation'].value_counts())
train_df.head(5)

is a          36776
said in       29667
support       16054
attack         2408
equivalent      667
Name: relation, dtype: int64


,head,relation,tail
0,well over a million and a quarter americans ar...,said in,11-10-1992
1,i've already made a proposal,said in,25-09-1988
2,i would support an effort to ban corporate sof...,is a,premise
3,"we're making some progress, doing a little bet...",is a,claim
4,"when it comes to our national security, i mean...",said in,16-10-2012


In [20]:
print(test_df['relation'].value_counts())
test_df.head()

support    1827
attack      252
Name: relation, dtype: int64


,head,relation,tail
0,america must be able to grow enough not only t...,support,that isn't good enough
1,it's the people listening to this broadcast,support,"hen you do well, america does well"
2,what he's quoting is not the senate budget com...,attack,he'll either have to raise your taxes by $900 ...
3,this is exactly the kind of track record we've...,support,we have to deal with zarqawi by taking him out
4,it's time to change,support,i want to bring that change to the american pe...


In [21]:
print(dev_df['relation'].value_counts())
dev_df.head()

support    1945
attack      206
Name: relation, dtype: int64


,head,relation,tail
0,"we want to invest in the creative, innovative ...",support,we have a plan
1,"they see, when we say that these options are o...",support,let's look at this from the view of the ayatol...
2,he said those successes are fragile,support,if we set a specific date for withdrawal -- an...
3,they'll welcome us,support,i don't think they'll look at us with envy
4,"in the congress, in the house of representativ...",support,i worked hard to learn the subject of nuclear ...


In [22]:
train_df.to_csv(f"{DRIVE_URL}/kge_dataset/triplets_file_train.tsv", index=False)
dev_df.to_csv(f"{DRIVE_URL}/kge_dataset/triplets_file_val.tsv", index=False)
test_df.to_csv(f"{DRIVE_URL}/kge_dataset/triplets_file_test.tsv", index=False)

In [23]:
triplets = pd.concat([test_df, pd.concat([train_df, dev_df], axis=0)], axis=0)
triplets.to_csv(f"{DRIVE_URL}/kge_dataset/triplets_file.tsv", index=False)

In [24]:
original = triplets[(triplets['relation']!= 'is a') & (triplets['relation']!= 'said in')]
original.to_csv(f"{DRIVE_URL}/kge_dataset/original_triplets_file.tsv", index=False)